<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L07-calibrated-judges/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L07-calibrated-judges/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L07-calibrated-judges/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L07-calibrated-judges/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L07 · Judges that agree with humans: calibrating an LLM-as-judge with bias probes and bias-corrected scores

**You will build:** the instruments that turn "a model said A is better" into a number you can
defend: a swap-order probe for position bias, a swap-resolved verdict, a length probe, and a
bias-corrected win rate with an interval, all measured against pairs that humans labelled.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, numpy only · **Prerequisites:** T10-L06-eval-power-and-ranks,
T07-L03-evaluating-agents, P02-L06-review-queues-real-reviewers

The judge here is a **scripted stand-in** with biases that this lesson plants on purpose, so every
number is checkable against the truth. The lesson measures a *method*; it makes no claim about how
any real judge model behaves.

By the end you will be able to:

1. Implement a swap-order probe, and measure how often a judge takes the first slot in both orders.
2. Implement a swap-resolved verdict, and measure what it fixes and what it leaves in place.
3. Implement a length probe that compares the judge with humans on the same pairs.
4. Implement a prevalence-corrected win rate from the judge's measured sensitivity and specificity.
5. Implement a paired bootstrap interval for that win rate, and measure when it covers the target.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T10-L06`, `T07-L03`, `P02-L06`

**This lesson assumes you can already:**

- use numpy boolean arrays: compare, combine with & and ~, and take a mean or a count over a mask
- read Cohen's kappa as agreement between two raters corrected for chance (P02-L06 builds it; this lesson reuses it)
- recognise a pairwise judge: it is shown two answers to one prompt and asked which is better

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685) · Zheng et al. (arXiv 2306.05685) · *free* — the paper that names position, verbosity and self-enhancement bias and proposes the swap-order fix used in exercises 1 and 2
- [Length-Controlled AlpacaEval, a simple way to debias automatic evaluators](https://arxiv.org/abs/2404.04475) · Dubois, Galambosi, Liang and Hashimoto (arXiv 2404.04475) · *free* — a worked account of a judge's length preference and a regression that controls for it, the next step after exercise 3
- [Efficient Inference for Noisy LLM-as-a-Judge Evaluation](https://arxiv.org/abs/2601.05420) · Chen et al., arXiv 2601.05420 · *free* — compares Rogan-Gladen-style correction, the method of exercises 4 and 5, with prediction-powered inference on LLM-judge win rates
- [sklearn.metrics.cohen_kappa_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html) · scikit-learn developers (official documentation) · *free* — the reference definition of the kappa that section 2 prints, to check your reading of the numbers
- [NumPy, the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · The NumPy project · *free* — boolean masks, indexing and means, the few numpy operations every exercise here is written in

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import sys
from dataclasses import dataclass
from typing import Callable, Sequence

import numpy as np

print("python", sys.version.split()[0])
print("numpy ", np.__version__)

# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 60     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---

# Every check and every demo below runs through `_try`, so pressing Run all before you have written
# a line reaches the progress board at the foot of the notebook instead of stopping at the first stub.
_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, function, where).
_EXERCISES = [("exercise 1", "swap_counts", "section 3"),
              ("exercise 2", "swap_resolved", "section 4"),
              ("exercise 3", "longer_win_rate", "section 5"),
              ("exercise 4", "corrected_win_rate", "section 6"),
              ("exercise 5", "bootstrap_interval", "section 7")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The function that raised NotImplementedError: the innermost frame of the traceback."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (`longer_win_rate`)"; several exercises -> "exercises 3 and 4"."""
    if len(labels) > 1 and all(label in _FUNCTION for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} (`{_FUNCTION[label]}`)" if label in _FUNCTION else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken exercise
    never hides the feedback on the others. A demo names the exercises it `needs`: until each has
    passed its check, the demo says which one it is waiting for and skips. Nothing is swallowed:
    every outcome is recorded for the progress board, and a script run exits non-zero there if any
    check came back wrong. A check that runs longer than its limit is stopped and counts as failed,
    so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _FUNCTION:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The setting: a judge you cannot take on trust

A **judge** is a model asked "which of these two answers is better?". It is cheap, so teams use it to
score thousands of pairs that no human will ever read. The trouble is that a judge answers two
questions at once: *which answer is better*, and *which slot, which length, whose family*. A score
built on it inherits every habit it has.

This lesson measures those habits, and then corrects for them. The world is a simulation, so the truth
is known:

* a **candidate** model is compared with a **baseline** on many prompts, and the candidate's position
  (first slot or second) is chosen at random for each pair;
* **humans** labelled a few pairs; the rest only ever get the judge's verdict. The simulation also keeps
  the humans' verdicts and the planted truth for those unlabelled pairs, **to score the method with**;
  in a real project you would not have them;
* the **judge is a scripted stand-in** with three planted biases: it leans towards the first slot, towards
  the longer answer, and towards the candidate (the judge belongs to the candidate's family). Their
  strengths are printed in the next cell. This measures a *method*, not any real judge model.

The agreement score between a judge and humans is **Cohen's kappa**, built in P02-L06. You do not build
it again: a faithful copy is in the next cell, given.

In [ ]:
SEED = 20261019                 # the date this lesson was written; no seed was searched
N_LAB, N_UNL = 400, 4000        # pairs that humans labelled / pairs that only the judge sees
BOOT = 200                      # bootstrap replicates per interval
REPLICATES = 100                # fresh worlds, for the coverage experiment in section 7
COV_BOOT, COV_UNL = 50, 1000    # the coverage experiment repeats the interval 200 times, so each one is smaller

# The planted biases and noise. These are parameters of this simulation, not claims about real judges.
BIAS_POSITION = 0.5             # added to the score of whatever sits in the first slot
BIAS_LENGTH = 1.0               # per unit of log(length of the first answer / length of the second)
BIAS_SELF = 0.3                 # added to the candidate's side: the judge is from the candidate's family
JUDGE_NOISE = 0.6               # the judge's own wobble, drawn afresh for every call
HUMAN_NOISE = 0.3               # humans are not perfect either
CANDIDATE_EDGE = 0.25           # the candidate's average quality advantage over the baseline
LENGTH_PER_QUALITY = 0.15       # better answers are a little longer, honestly
LENGTH_SPREAD = 0.35            # spread of an answer's log-length that has nothing to do with quality
CANDIDATE_LONGER = 0.20         # the candidate writes longer answers (log scale) ...
CANDIDATE_LONGER_SHIFTED = 0.50     # ... and in the shifted world, much longer


@dataclass
class Pairs:
    """A set of answer pairs. Slot 1 and slot 2 are the positions the judge saw in the ORIGINAL order."""
    cand_first: np.ndarray          # bool: the candidate's answer sits in slot 1
    len_first: np.ndarray           # tokens in slot 1's answer
    len_second: np.ndarray          # tokens in slot 2's answer
    pick_first: np.ndarray          # bool: the judge picked slot 1 (original order)
    pick_first_swapped: np.ndarray  # bool: the judge picked slot 1 when the same pair was shown swapped
    human_cand: np.ndarray          # bool: humans say the candidate is better
    truth_cand: np.ndarray          # bool: the candidate really is better (planted; no real project has it)

    @property
    def judge_cand(self) -> np.ndarray:
        """bool: the judge says the candidate is better (original order)."""
        return self.pick_first == self.cand_first

    @property
    def human_first(self) -> np.ndarray:
        """bool: humans say the answer in slot 1 is better."""
        return self.human_cand == self.cand_first

    def __len__(self) -> int:
        return len(self.cand_first)


def judge_scores(q_first, q_second, len_first, len_second, cand_first, noise, position=BIAS_POSITION):
    """The scripted judge's score for slot 1 over slot 2. It picks slot 1 when this is above zero."""
    return ((q_first - q_second) + position + BIAS_LENGTH * np.log(len_first / len_second)
            + BIAS_SELF * np.where(cand_first, 1.0, -1.0) + JUDGE_NOISE * noise)


def make_pairs(rng, n, longer=CANDIDATE_LONGER, position=BIAS_POSITION) -> Pairs:
    """Draw n pairs. Every random number is drawn in a fixed order, so a world depends only on the seed."""
    q_cand = rng.normal(CANDIDATE_EDGE, 1.0, n)
    q_base = rng.normal(0.0, 1.0, n)
    z_cand, z_base = rng.normal(size=n), rng.normal(size=n)
    cand_first = rng.random(n) < 0.5
    human_noise, noise_1, noise_2 = rng.normal(size=n), rng.normal(size=n), rng.normal(size=n)
    l_cand = np.rint(150 * np.exp(longer + LENGTH_PER_QUALITY * q_cand + LENGTH_SPREAD * z_cand)).astype(int)
    l_base = np.rint(150 * np.exp(LENGTH_PER_QUALITY * q_base + LENGTH_SPREAD * z_base)).astype(int)
    q1, q2 = np.where(cand_first, q_cand, q_base), np.where(cand_first, q_base, q_cand)
    l1, l2 = np.where(cand_first, l_cand, l_base), np.where(cand_first, l_base, l_cand)
    pick_first = judge_scores(q1, q2, l1, l2, cand_first, noise_1, position) > 0
    # the swapped call: the slots trade places, the judge's wobble is a fresh draw
    pick_swapped = judge_scores(q2, q1, l2, l1, ~cand_first, noise_2, position) > 0
    return Pairs(cand_first=cand_first, len_first=l1, len_second=l2, pick_first=pick_first,
                 pick_first_swapped=pick_swapped, human_cand=(q_cand - q_base + HUMAN_NOISE * human_noise) > 0,
                 truth_cand=q_cand > q_base)


# Given, from P02-L06 (a faithful copy; read its exercise 2 for the derivation).
def raw_agreement(labels_a: Sequence[str], labels_b: Sequence[str]) -> float:
    """The fraction of paired items on which the two raters gave the same label."""
    a, b = list(labels_a), list(labels_b)
    if len(a) != len(b):
        raise ValueError(f"the two raters labelled {len(a)} and {len(b)} items; agreement needs the same items")
    if not a:
        raise ValueError("agreement over no items is undefined")
    return sum(x == y for x, y in zip(a, b)) / len(a)


def chance_agreement(labels_a: Sequence[str], labels_b: Sequence[str]) -> float:
    """p_e: the agreement two raters would reach by labelling at random with their own label frequencies."""
    a, b = list(labels_a), list(labels_b)
    if len(a) != len(b):
        raise ValueError(f"the two raters labelled {len(a)} and {len(b)} items; agreement needs the same items")
    if not a:
        raise ValueError("agreement over no items is undefined")
    n = len(a)
    return sum((a.count(label) / n) * (b.count(label) / n) for label in sorted(set(a) | set(b)))


def cohens_kappa(labels_a: Sequence[str], labels_b: Sequence[str]) -> float:
    """Cohen's kappa: (p_o - p_e) / (1 - p_e); nan when both raters used one and the same single label."""
    p_o, p_e = raw_agreement(labels_a, labels_b), chance_agreement(labels_a, labels_b)
    if 1.0 - p_e < 1e-12:
        return float("nan")
    return (p_o - p_e) / (1.0 - p_e)


def verdicts(cand_wins) -> list:
    """bool array -> the labels kappa compares."""
    return ["candidate" if v else "baseline" for v in cand_wins]


_world = np.random.default_rng(SEED)
LAB = make_pairs(_world, N_LAB)                                 # humans labelled these
UNL = make_pairs(_world, N_UNL)                                 # only the judge sees these (humans held back to score the method)
SHIFTED = make_pairs(_world, N_UNL, longer=CANDIDATE_LONGER_SHIFTED)     # as UNL, but the candidate is far more verbose

if __name__ == "__main__":
    print(f"{N_LAB} labelled pairs, {N_UNL} pairs only the judge sees, one more set of {N_UNL} where the candidate is far more verbose")
    print(f"planted judge: position {BIAS_POSITION}, length {BIAS_LENGTH}, self-preference {BIAS_SELF}, "
          f"noise {JUDGE_NOISE}; planted human noise {HUMAN_NOISE}")
    print(f"candidate in the first slot on {LAB.cand_first.mean():.1%} of the labelled pairs (slots are randomised)")

## 2. Agreement first, and why it is not enough

The first thing anyone computes is agreement with humans. The next cell does it on the labelled
pairs, for the judge in the original order and in the swapped order, then looks at the thing the
judge is *used* for: the candidate's win rate on the pairs only it sees.

Read the kappa and then the win rates: a judge can agree with humans and still move a leaderboard.

In [ ]:
def _show_agreement() -> None:
    human = verdicts(LAB.human_cand)
    in_order = verdicts(LAB.judge_cand)
    swapped = verdicts(LAB.pick_first_swapped != LAB.cand_first)     # the candidate is in slot 1 iff cand_first is False
    print(f"raw agreement with humans, original order: {raw_agreement(human, in_order):.3f}   "
          f"kappa {cohens_kappa(human, in_order):.3f}")
    print(f"raw agreement with humans, swapped order:  {raw_agreement(human, swapped):.3f}   "
          f"kappa {cohens_kappa(human, swapped):.3f}")
    print(f"\ncandidate win rate on the {N_UNL} pairs only the judge sees:")
    print(f"  planted truth            {UNL.truth_cand.mean():.3f}")
    print(f"  humans (held back)       {UNL.human_cand.mean():.3f}")
    print(f"  judge, original order    {UNL.judge_cand.mean():.3f}")
    print(f"  judge, swapped order     {(UNL.pick_first_swapped != UNL.cand_first).mean():.3f}")
    print(f"the judge's score is {UNL.judge_cand.mean() - UNL.human_cand.mean():+.3f} away from the humans'; "
          "the rest of the lesson finds out why, and what to do about it.")


if __name__ == "__main__":
    _show_agreement()

## 3. Exercise 1 — `swap_counts`: does the slot move the verdict?

The standard probe for **position bias** is to ask the judge twice, with the two answers in opposite
slots. If it judges content, it picks the *same answer* both times. If it picks the first slot both
times, or the second slot both times, the slot decided it.

`first_a[i]` is True when the judge picked slot 1 in the original order; `first_b[i]` is True when it
picked slot 1 in the **swapped** order (where slot 1 holds the other answer). Return three integers:
how many pairs it picked slot 1 in both orders, how many it picked slot 2 in both orders, and how many it
picked the same *answer* in both orders.

<details><summary>💡 Hint 1 — what to think about</summary>

When the answers swap places, "slot 1" means a different answer. So picking slot 1 and then slot 2 is
the judge choosing the same answer twice. Work out, for each of the four combinations of the two
verdicts, which of your three counts it belongs to.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Treat the two arrays as booleans. Slot 1 both times is "both True". Slot 2 both times is "both False".
The same answer both times is the case where the two verdicts differ. The three counts add up to the
number of pairs. Different lengths are a mistake in the caller: raise `ValueError`.

</details>

In [ ]:
def swap_counts(first_a, first_b):
    """Count how the judge's verdicts relate across the two orders.

    Args:
        first_a: bool array, True where the judge picked slot 1 in the original order.
        first_b: bool array, True where it picked slot 1 in the swapped order.

    Different shapes, or arrays that are not one-dimensional, raise ``ValueError``. Zero pairs give ``(0, 0, 0)``.

    Returns:
        ``(first_both, second_both, consistent)`` as Python ints: pairs where it picked slot 1
        both times, slot 2 both times, and the SAME ANSWER both times.

    Example:
        >>> swap_counts(np.array([True, True, False, False]), np.array([True, False, True, False]))
        (1, 1, 2)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_swap_counts() -> None:
    got = swap_counts(np.array([True, True, False, False, True]), np.array([True, False, True, False, False]))
    assert tuple(got) == (1, 1, 3), (
        f"got {tuple(got)}. A pair is consistent when the judge picks the same ANSWER in both orders, and "
        "because the slots swapped that means DIFFERENT slots: (slot 1, then slot 2) or (slot 2, then slot 1).")
    assert all(type(v) is int for v in got), f"return Python ints, not {[type(v).__name__ for v in got]}"
    f, s, c = swap_counts(np.array([True, True, True]), np.array([True, True, True]))
    assert (f, s, c) == (3, 0, 0), f"a judge that always says slot 1 has first_both = n and nothing else; got {(f, s, c)}"
    try:
        swap_counts(np.array([True]), np.array([True, False]))
    except ValueError:
        pass
    else:
        raise AssertionError("two orders with different numbers of pairs must raise ValueError")
    print("exercise 1 looks right")


_try("exercise 1", _check_swap_counts)

In [ ]:
def _show_position_bias() -> None:
    n = len(LAB)
    first_both, second_both, consistent = swap_counts(LAB.pick_first, LAB.pick_first_swapped)
    print(f"over {n} labelled pairs: slot 1 both times {first_both}, slot 2 both times {second_both}, "
          f"the same answer both times {consistent}")
    print(f"  position bias, (slot-1-both minus slot-2-both) / pairs: {(first_both - second_both) / n:+.3f}")
    plain = make_pairs(np.random.default_rng(SEED), N_LAB, position=0.0)      # the same draws as LAB, no position bias
    f0, s0, c0 = swap_counts(plain.pick_first, plain.pick_first_swapped)
    print(f"the same world with the position bias switched off (only a simulation can): slot 1 both times {f0}, "
          f"slot 2 both times {s0}, same answer {c0}; bias {(f0 - s0) / n:+.3f}")
    print(f"slot-1-both minus slot-2-both is {first_both - second_both:+d} here and {f0 - s0:+d} with the bias off: "
          f"the difference between the two, {(first_both - second_both) - (f0 - s0):+d} pairs, is what the planted slot bias moved")


_try("position bias", _show_position_bias, needs=("exercise 1",))

## 4. Exercise 2 — `swap_resolved`: a verdict that survives the swap

MT-Bench's authors describe the conservative fix (quoted in `claims.yaml`): call the judge twice with the
order swapped, and declare a win only when the same answer is preferred in both orders; otherwise call it
a tie. Write that as an array.

Return one integer per pair, in terms of the ORIGINAL order's slots: **+1** when the answer in slot 1
(original order) won both times, **−1** when the answer in slot 2 (original order) won both times, **0**
when the judge was inconsistent. Take care over which swapped verdict goes with which sign.

<details><summary>💡 Hint 1 — what to think about</summary>

In the swapped order the answer that was in slot 2 is now in slot 1. Picking slot 1 there is a vote for
the original slot 2. Write down, for each combination of the two verdicts, who got a vote from whom.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

+1 needs a vote for the original slot 1 in the original order AND in the swapped order, which means
"first" in the first and "second" in the second. −1 is the mirror image. Everything else is 0. Return an
integer array, not booleans: the demos compare it with +1 and −1.

</details>

In [ ]:
def swap_resolved(first_a, first_b):
    """Resolve the two orders into one verdict per pair, in the original order's slots.

    Args:
        first_a, first_b: as in `swap_counts`.

    Different shapes raise ``ValueError``. Zero pairs give an empty array.

    Returns:
        An array of +1, -1 and 0: +1 where the answer in the original slot 1 won in both orders,
        -1 where the answer in the original slot 2 won in both orders, 0 where the judge was
        inconsistent.

    Example:
        >>> swap_resolved(np.array([True, True, False, False]), np.array([True, False, True, False])).tolist()
        [0, 1, -1, 0]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_swap_resolved() -> None:
    got = np.asarray(swap_resolved(np.array([True, True, False, False]), np.array([True, False, True, False])))
    assert got.tolist() == [0, 1, -1, 0], (
        f"got {got.tolist()}. (slot 1, slot 1) and (slot 2, slot 2) are the judge following the slot: 0. "
        "(slot 1, slot 2) is the original slot-1 answer winning twice: +1. (slot 2, slot 1) is -1.")
    assert got.dtype.kind in "iu" or got.dtype.kind == "f", "return numbers (+1, -1, 0), not booleans"
    try:
        swap_resolved(np.array([True]), np.array([True, False]))
    except ValueError:
        pass
    else:
        raise AssertionError("two orders with different numbers of pairs must raise ValueError")
    print("exercise 2 looks right")


_try("exercise 2", _check_swap_resolved)

In [ ]:
def _show_swap_resolved() -> None:
    res = np.asarray(swap_resolved(LAB.pick_first, LAB.pick_first_swapped))
    decisive = res != 0
    human = LAB.human_cand
    cand_by_res = (res == 1) == LAB.cand_first             # on a decisive pair: the candidate won both times
    print(f"the swap-resolved judge is decisive on {decisive.sum()} of {len(LAB)} pairs and abstains on "
          f"{(~decisive).sum()} ({(~decisive).mean():.1%})")
    print(f"kappa with humans, all {len(LAB)} pairs, original order:   "
          f"{cohens_kappa(verdicts(human), verdicts(LAB.judge_cand)):.3f}")
    print(f"kappa with humans, the {decisive.sum()} decisive pairs:          "
          f"{cohens_kappa(verdicts(human[decisive]), verdicts(cand_by_res[decisive])):.3f}")
    agree_dec, agree_abs = (LAB.judge_cand == human)[decisive].mean(), (LAB.judge_cand == human)[~decisive].mean()
    print(f"the original-order verdict agrees with humans on {agree_dec:.3f} of the decisive pairs and on {agree_abs:.3f} "
          f"of the abstained ones: the abstentions are {'the pairs where the verdict was worth less' if agree_abs < agree_dec else 'no worse'}")
    print("and the win rate it gives:")
    print(f"  humans on all pairs {human.mean():.3f} | resolved judge on its decisive pairs "
          f"{cand_by_res[decisive].mean():.3f} | humans on those same pairs {human[decisive].mean():.3f}")


_try("swap resolved", _show_swap_resolved, needs=("exercise 2",))

## 5. Exercise 3 — `longer_win_rate`: the probe swapping cannot do

Swapping moves *slots*. A length preference moves with the answer, so the swap cannot see it: the longer
answer is still longer in the second call. The probe is to ask how often the winner is the **longer**
answer, for the judge and for humans **on the same pairs**. In this world longer answers are slightly
better by construction, so humans lean towards the longer one too; their rate is the baseline, and the
gap between the judge's rate and theirs is the judge's length bias.

Write `longer_win_rate(first_wins, len_first, len_second)`: among the pairs whose two answers differ in
length, the share whose winner is the longer one. Pairs of equal length have no longer answer and are
left out of numerator and denominator. If every pair is of equal length there is nothing to measure:
raise `ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

For each pair, decide whether the winner is the longer answer. The winner is in slot 1 or slot 2
according to `first_wins`; the longer answer is in slot 1 or slot 2 according to the two lengths. Then
ask what to do with a pair where neither is longer.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Build a boolean "slot 1 is longer" array, compare it with `first_wins`, and average the comparison over
only the pairs whose lengths differ. A winner in slot 2 on a pair where slot 2 is longer counts too.

</details>

In [ ]:
def longer_win_rate(first_wins, len_first, len_second):
    """The share of pairs, among those whose answers differ in length, won by the longer answer.

    Args:
        first_wins: bool array, True where the answer in slot 1 won.
        len_first, len_second: integer arrays, the length of the answer in each slot.

    Arrays of different shapes, or no pair whose lengths differ, raise ``ValueError``.

    Returns:
        A number in [0, 1].

    Example:
        >>> longer_win_rate(np.array([True, False, True]), np.array([10, 10, 5]), np.array([5, 20, 5]))
        1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_longer_win_rate() -> None:
    got = longer_win_rate(np.array([True, False, True]), np.array([10, 10, 5]), np.array([5, 20, 5]))
    assert abs(got - 1.0) < 1e-12, (
        f"got {got!r}. The third pair has two answers of equal length: it has no longer answer, so it counts "
        "in neither the numerator nor the denominator. The first two pairs are both won by the longer answer.")
    got = longer_win_rate(np.array([True, True]), np.array([10, 5]), np.array([5, 10]))
    assert abs(got - 0.5) < 1e-12, f"got {got!r}; one winner is the longer answer and one is the shorter: 0.5"
    try:
        longer_win_rate(np.array([True, False]), np.array([7, 7]), np.array([7, 7]))
    except ValueError:
        pass
    else:
        raise AssertionError("when no pair differs in length there is nothing to measure: raise ValueError")
    print("exercise 3 looks right")


_try("exercise 3", _check_longer_win_rate)

In [ ]:
def _show_length_gap() -> None:
    human = longer_win_rate(LAB.human_first, LAB.len_first, LAB.len_second)
    judge = longer_win_rate(LAB.pick_first, LAB.len_first, LAB.len_second)
    res = np.asarray(swap_resolved(LAB.pick_first, LAB.pick_first_swapped))
    d = res != 0
    swapped = longer_win_rate(res[d] == 1, LAB.len_first[d], LAB.len_second[d])
    human_d = longer_win_rate(LAB.human_first[d], LAB.len_first[d], LAB.len_second[d])
    print("how often the winner is the longer answer, on the labelled pairs:")
    print(f"  humans                          {human:.3f}")
    print(f"  judge, original order           {judge:.3f}   (gap to humans {judge - human:+.3f})")
    print(f"  judge, swap-resolved decisive   {swapped:.3f}   (humans on those pairs {human_d:.3f}; gap {swapped - human_d:+.3f})")
    before, after = judge - human, swapped - human_d
    print(f"the gap to humans was {before:+.3f} before swap-resolution (all {len(LAB)} pairs) and is {after:+.3f} after (the {int(d.sum())} decisive pairs); "
          f"{'most of that gap is still there (more than half of it)' if before > 0 and after > before / 2 else 'a positive gap that is no more than half the earlier one is left' if before > 0 and after > 0 else 'no positive gap is left'}: "
          "the two gaps are on different pair sets, so compare them as a guide only.")


_try("length gap", _show_length_gap, needs=("exercise 2", "exercise 3"))

## 6. Exercise 4 — `corrected_win_rate`: undo the judge's error on the pairs only it sees

You cannot remove the judge's biases, but you can **measure** how it errs on pairs humans labelled and
undo that on pairs they did not. Call "the candidate wins" the positive class. On the labelled pairs
measure the judge's **sensitivity** (of the pairs humans say the candidate wins, the share the judge also
says it wins) and **specificity** (of the pairs humans say the baseline wins, the share the judge says the
baseline wins). If the judge says "candidate" on a share `p_obs` of the unlabelled pairs, then

`p_obs = sensitivity * p + (1 - specificity) * (1 - p)`, so `p = (p_obs + specificity - 1) / (sensitivity + specificity - 1)`.

This is the Rogan–Gladen adjustment for a screening test (sourced in `claims.yaml`; a 2026 paper applies
the same idea to LLM judges). Return `(sensitivity, specificity, p)`, with `p` **clipped to [0, 1]**
because sampling noise can push the algebra just outside. Raise `ValueError` when the calibration pairs
differ in length or lack humans' positives or negatives, when `judge_unl` is empty, or when
`sensitivity + specificity` is **not above 1**, comparing the float sum with 1.0 as it stands, with no tolerance (a judge no better than a coin cannot be inverted).

<details><summary>💡 Hint 1 — what to think about</summary>

Sensitivity and specificity condition on what the HUMANS said, not on what the judge said. Which array
decides who is in the denominator? And when `sensitivity + specificity` is exactly 1, what happens to the
divisor?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Split the calibration pairs by the humans' label. Sensitivity is the mean of the judge's "candidate" over
the human-positive pairs; specificity is the mean of the judge's "baseline" over the human-negative pairs.
Take the judge's mean over the unlabelled pairs, apply the formula in the text, then clip. Check the
guards before dividing.

</details>

In [ ]:
def corrected_win_rate(judge_cal, human_cal, judge_unl):
    """Rogan-Gladen corrected win rate.

    Args:
        judge_cal, human_cal: bool arrays on the calibration pairs, True = "the candidate wins",
            by the judge and by humans, for the same pairs in the same order.
        judge_unl: bool array, the judge's verdicts on the pairs humans did not label.

    Raises ``ValueError`` for calibration arrays of different lengths, no human-positive or no
    human-negative pair, an empty ``judge_unl``, or ``sensitivity + specificity <= 1``.

    Returns:
        ``(sensitivity, specificity, p)`` as Python floats, with ``p`` clipped into [0, 1].

    Example (humans: 2 of 4 positive; the judge gets 1 positive and both negatives right):
        >>> corrected_win_rate(np.array([True, False, False, False]), np.array([True, True, False, False]),
        ...                    np.array([True, False]))
        (0.5, 1.0, 1.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_corrected_win_rate() -> None:
    human = np.array([True] * 10 + [False] * 10)
    judge = np.array([True] * 8 + [False] * 2 + [True] * 1 + [False] * 9)       # 8/10 and 9/10
    unl = np.array([True] * 31 + [False] * 69)
    se, sp, p = corrected_win_rate(judge, human, unl)
    assert abs(se - 0.8) < 1e-12 and abs(sp - 0.9) < 1e-12, (
        f"got sensitivity {se!r}, specificity {sp!r}. Both are measured against the HUMANS' label: "
        "sensitivity is the judge's hit rate on human-positive pairs, specificity its hit rate on human-negative ones.")
    assert abs(p - 0.3) < 1e-12, f"got p = {p!r}; solve p_obs = se*p + (1 - sp)*(1 - p) for p"
    assert type(se) is float and type(sp) is float and type(p) is float, (
        f"got types ({type(se).__name__}, {type(sp).__name__}, {type(p).__name__}): return Python floats (wrap each with float(), "
        "NumPy's np.float64 is a different type)")
    _, _, hi = corrected_win_rate(judge, human, np.array([True] * 100))
    assert hi == 1.0, f"an observed rate of 1.0 gives an algebraic value above 1; clip it to 1.0, got {hi!r}"
    try:
        corrected_win_rate(np.array([True, False, True, False]), np.array([True, True, False, False]), unl)
    except ValueError:
        pass
    else:
        raise AssertionError("sensitivity + specificity of exactly 1 (a coin) cannot be inverted: raise ValueError")
    print("exercise 4 looks right")


_try("exercise 4", _check_corrected_win_rate)

In [ ]:
def _show_correction() -> None:
    se, sp, p = corrected_win_rate(LAB.judge_cand, LAB.human_cand, UNL.judge_cand)
    print(f"on the {N_LAB} labelled pairs the judge has sensitivity {se:.3f} and specificity {sp:.3f}")
    print(f"on the {N_UNL} pairs only it sees, the candidate's win rate is:")
    print(f"  raw judge        {UNL.judge_cand.mean():.3f}   (error against the humans' rate {UNL.judge_cand.mean() - UNL.human_cand.mean():+.3f})")
    print(f"  corrected judge  {p:.3f}   (error {p - UNL.human_cand.mean():+.3f})")
    print(f"  humans (held back) {UNL.human_cand.mean():.3f}   planted truth {UNL.truth_cand.mean():.3f}")
    h = UNL.human_cand.mean()
    print(f"the corrected rate is {'closer to' if abs(p - h) < abs(UNL.judge_cand.mean() - h) else 'no closer to'} the humans' rate than the raw one; "
          f"it aims at what humans would say, and the humans are noisy too, {abs(h - UNL.truth_cand.mean()):.3f} from the planted truth.")


_try("correction", _show_correction, needs=("exercise 4",))

## 7. Exercise 5 — `bootstrap_interval`: how sure is the corrected number?

A corrected rate built from a few hundred labelled pairs is itself an estimate, and the correction divides by a
number that can be small, so it can be noisy. The honest report is an interval.

The **bootstrap** redraws pairs with replacement many times and recomputes the estimate each time. The
resampling indices are given: replicate `b` uses `idx_cal[b]` for the calibration pairs and `idx_unl[b]`
for the unlabelled pairs. Two rules matter. A calibration pair is a *pair*: take the judge's verdict AND
the humans' verdict at the same index. And recompute the **whole** corrected estimate in every replicate,
sensitivity and specificity included, clipped to [0, 1] exactly as in exercise 4. Return the 2.5th and 97.5th percentiles of the replicate estimates
(numpy's default linear interpolation).

<details><summary>💡 Hint 1 — what to think about</summary>

What does a replicate need as inputs, and what does your function from exercise 4 already compute from
them? Think about what would go wrong if the judge's verdicts and the humans' verdicts were resampled with
different indices.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

For each row of the two index matrices, index the three arrays with that row, get the corrected estimate,
and keep it. Take the two percentiles of the list.

</details>

In [ ]:
def bootstrap_interval(judge_cal, human_cal, judge_unl, idx_cal, idx_unl):
    """A 95% bootstrap interval for the corrected win rate.

    Args:
        judge_cal, human_cal, judge_unl: as in `corrected_win_rate`.
        idx_cal: integer array of shape (B, number of calibration pairs); row b picks replicate b's
            calibration pairs, for the judge's AND the humans' verdicts together.
        idx_unl: integer array of shape (B, number of unlabelled pairs).

    ``idx_cal`` and ``idx_unl`` with different numbers of rows raise ``ValueError``.

    Returns:
        ``(low, high)``: the 2.5th and 97.5th percentiles of the B corrected estimates (each one clipped to
        [0, 1], as `corrected_win_rate` returns it; numpy's
        default linear interpolation), as Python floats.

    Example (a judge that is always right, so each replicate is the unlabelled mean of its draw):
        >>> bootstrap_interval(np.array([True, False]), np.array([True, False]), np.array([True, False, False, False]),
        ...                    np.array([[0, 1], [0, 1]]), np.array([[0, 1, 2, 3], [0, 0, 0, 0]]))
        (0.26875, 0.98125)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bootstrap_interval() -> None:
    judge = human = np.array([True, False])
    unl = np.array([True, False, False, False])
    cal_rows = np.tile(np.arange(2), (5, 1))                                   # every replicate keeps both calibration pairs
    unl_rows = np.array([[0, 1, 2, 3], [0, 0, 1, 2], [1, 2, 3, 3], [0, 0, 0, 1], [0, 0, 0, 0]])
    lo, hi = bootstrap_interval(judge, human, unl, cal_rows, unl_rows)       # replicate estimates 0.25, 0.5, 0, 0.75, 1
    assert abs(lo - 0.025) < 1e-12 and abs(hi - 0.975) < 1e-12, (
        f"got ({lo!r}, {hi!r}). With a perfect judge each replicate's estimate is the share of True verdicts in "
        "its unlabelled draw; take the 2.5th and 97.5th percentiles of those five numbers (linear interpolation).")
    assert type(lo) is float and type(hi) is float, (
        f"got types ({type(lo).__name__}, {type(hi).__name__}): return Python floats (wrap each with float(), "
        "NumPy's np.float64 is a different type)")
    print("exercise 5 looks right")


_try("exercise 5", _check_bootstrap_interval, needs=("exercise 4",))

In [ ]:
def draw_indices(rng, n_cal, n_unl, b=BOOT):
    """Given: the resampling indices for one interval."""
    return rng.integers(0, n_cal, (b, n_cal)), rng.integers(0, n_unl, (b, n_unl))


def _show_interval() -> None:
    rng = np.random.default_rng(SEED + 1)
    ic, iu = draw_indices(rng, N_LAB, N_UNL)
    lo, hi = bootstrap_interval(LAB.judge_cand, LAB.human_cand, UNL.judge_cand, ic, iu)
    p = corrected_win_rate(LAB.judge_cand, LAB.human_cand, UNL.judge_cand)[2]
    print(f"corrected win rate {p:.3f}, 95% interval ({lo:.3f}, {hi:.3f}); the humans' rate on those pairs {UNL.human_cand.mean():.3f}; "
          f"raw judge {UNL.judge_cand.mean():.3f}")
    print(f"the raw judge is outside the interval: {not (lo <= UNL.judge_cand.mean() <= hi)}")


_try("interval", _show_interval, needs=("exercise 4", "exercise 5"))

### Does the interval cover? And when does the correction break?

A single interval proves nothing. The next cell repeats the whole procedure on fresh worlds: draw a
labelled set of the size above and a smaller unlabelled one (the next cell prints both sizes), correct, bootstrap, and count how often the interval holds the
humans' win rate on that world's unlabelled pairs. Then it does it again where the **unlabelled pairs
come from a different mix** than the calibration ones: the candidate has turned much more verbose, so the
judge's length habit is worth more than it was on the pairs where you measured it.

In [ ]:
def _coverage() -> None:
    rng = np.random.default_rng(SEED + 2)
    rows = {"same mix": CANDIDATE_LONGER, "shifted mix": CANDIDATE_LONGER_SHIFTED}
    held = {}
    for name, longer in rows.items():
        covered, raw_err, fix_err = 0, [], []
        for _ in range(REPLICATES):
            lab, unl = make_pairs(rng, N_LAB), make_pairs(rng, COV_UNL, longer=longer)
            ic, iu = draw_indices(rng, N_LAB, COV_UNL, COV_BOOT)
            lo, hi = bootstrap_interval(lab.judge_cand, lab.human_cand, unl.judge_cand, ic, iu)
            target = unl.human_cand.mean()
            covered += lo <= target <= hi
            raw_err.append(unl.judge_cand.mean() - target)
            fix_err.append(corrected_win_rate(lab.judge_cand, lab.human_cand, unl.judge_cand)[2] - target)
        held[name] = covered
        print(f"{name:12s} ({N_LAB} labelled, {COV_UNL} unlabelled pairs, {COV_BOOT} replicates) interval held the humans' rate in {covered} of {REPLICATES} worlds; "
              f"mean error raw {np.mean(raw_err):+.3f}, corrected {np.mean(fix_err):+.3f}")
    print("the interval held the target less often when the unlabelled pairs came from another mix: "
          f"{held['shifted mix'] < held['same mix']}")


_try("coverage", _coverage, needs=("exercise 4", "exercise 5"))

## 8. Self-check

1. In your swap probe a judge picks slot 1 in both orders on a large share of pairs, and slot 2 in both
   orders on almost none. What is the sound reading?
   - (a) It judges those pairs equal in quality, so it declines to choose.
   - (b) For those pairs the slot, not the content, moved the verdict.
   - (c) The humans who labelled them were inconsistent.
   - (d) It prefers the longer answer, which sits in slot 1 by chance.

2. After swap-resolution the judge's agreement with humans on its decisive pairs goes up. A colleague
   says its win rate for a very verbose candidate can now be trusted. What is wrong with that?
   - (a) Nothing: agreement is the only property that matters for a win rate.
   - (b) Swapping makes the judge noisier, so its verdicts are less reliable.
   - (c) The swap removes a slot effect; a preference for longer answers travels with the answer.
   - (d) Decisive pairs are a random sample, so their win rate is unbiased.

3. You correct a judge's win rate on unlabelled pairs using sensitivity and specificity measured on
   labelled ones. Which assumption carries the whole result?
   - (a) The unlabelled pairs resemble the labelled ones in what drives the judge's errors.
   - (b) The judge must have no bias on the labelled pairs.
   - (c) The humans must never disagree with each other.
   - (d) The unlabelled set must be exactly as large as the labelled set.

4. A bootstrap replicate resamples the judge's calibration verdicts and the humans' with two different
   sets of indices. What goes wrong?
   - (a) Nothing, because the two samples are independent anyway.
   - (b) The interval comes out too narrow, because the humans' labels are fixed.
   - (c) The replicate has fewer pairs than the original.
   - (d) It pairs a verdict on one pair with a label on another, so sensitivity and specificity drift to chance.

Answers come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
if __name__ == "__main__":
    print("facts for questions 3 and 4, printed here rather than taken on trust:")
    _lr = lambda P: float(np.mean(np.log(np.where(P.cand_first, P.len_first, P.len_second) / np.where(P.cand_first, P.len_second, P.len_first))))
    print(f"  measured: the candidate's mean log-length ratio is {_lr(UNL):+.2f} on the unlabelled pairs and {_lr(SHIFTED):+.2f} on the shifted ones, {_lr(SHIFTED) - _lr(UNL):+.2f} higher")
    _chance = np.random.default_rng(SEED + 3)
    _same = _chance.integers(0, N_LAB, N_LAB)
    _other = _chance.integers(0, N_LAB, N_LAB)
    print(f"  kappa of the judge with the humans on the same resampled pairs {cohens_kappa(verdicts(LAB.human_cand[_same]), verdicts(LAB.judge_cand[_same])):.3f}; "
          f"with the humans' labels taken from a different resample {cohens_kappa(verdicts(LAB.human_cand[_other]), verdicts(LAB.judge_cand[_same])):.3f}")

## 9. Common mistakes

* **Calling a high kappa "validated".** Agreement with humans is an average over pairs; it hides which
  pairs the judge gets wrong and why. The probes above split it by slot and by length.
* **Swapping once and stopping.** The swap-resolved verdict fixes the slot; the length
  preference survives it (section 5 prints the gap to humans before and after).
* **Resampling the judge and the humans separately.** A calibration pair is a pair.
* **Correcting with a calibration set from a different mix.** The shifted-mix row of section 7 is what
  it costs; the fix is a calibration set drawn from what you will score, or a model that conditions on
  what moved (length, for instance).
* **Forgetting the guard on the divisor.** `sensitivity + specificity` near 1 means the correction
  amplifies noise; at 1 or below it is undefined.

## 10. What this lesson does NOT cover

* **Real judges.** The judge is a script with planted biases. Real judges differ by model, prompt and
  version, and their biases can change with a release; the sources in `claims.yaml` are read as of
  19 October 2026. Re-measure yours.
* **Ties and scales.** Verdicts here are two-way. Ties, scores on a 1 to 10 scale and multi-turn
  conversations need their own agreement measures.
* **Regression-based length control.** AlpacaEval's length-controlled score fits a model that predicts the
  judge's preference from the length difference and reads off the prediction at zero difference; this lesson
  measures the bias and corrects a rate, it does not fit that model.
* **Fewer labels.** Prediction-powered inference and other estimators can use human labels more
  efficiently than the correction here; see the paper cited in `claims.yaml`.
* **How many labelled pairs.** T10-L06 covers how many items a comparison needs; here the number is fixed.

In [ ]:
def _recap() -> None:
    first_both, second_both, consistent = swap_counts(LAB.pick_first, LAB.pick_first_swapped)
    res = np.asarray(swap_resolved(LAB.pick_first, LAB.pick_first_swapped))
    se, sp, p = corrected_win_rate(LAB.judge_cand, LAB.human_cand, UNL.judge_cand)
    print("--- what you built, measured ---")
    print(f"  swap probe: slot 1 both times {first_both}, slot 2 both times {second_both}, same answer {consistent}; "
          f"swap-resolved abstentions {int((res == 0).sum())}")
    print(f"  length probe: judge picks the longer answer on {longer_win_rate(LAB.pick_first, LAB.len_first, LAB.len_second):.3f} "
          f"of the pairs that differ in length, humans on {longer_win_rate(LAB.human_first, LAB.len_first, LAB.len_second):.3f}")
    print(f"  corrected win rate {p:.3f} (raw {UNL.judge_cand.mean():.3f}, humans {UNL.human_cand.mean():.3f})")
    print("  every number above was computed by the code you just ran")


if __name__ == "__main__":
    _try("the closing recap", _recap, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "", function) else ""
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero, so a green exit code can never
    paper over it. In a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.